# einx Basics — worked examples (with plain numpy/torch equivalents)

Based on the official tutorial: https://einx.readthedocs.io/en/stable/gettingstarted/basics.html

**Mental model.** Every call looks like

```
out = einx.<op>("<input patterns>  ->  <output pattern>", *tensors)
```

- Each space-separated name is an axis. Matching names across patterns are the *same* axis.
- Brackets `[...]` mark the axes the *elementary* operation acts on (the axis that is
  summed, contracted, softmaxed, vmapped, ...). Everything **outside** brackets is
  *vectorized* — broadcast or looped over automatically.
- Axis order is **positional**: the name in slot *i* of an input pattern names that
  tensor's dimension *i*.

For each example below we show the einx form, print the result, then reproduce it with
ordinary numpy/torch so you can see exactly what einx expands to.

> Note: the docs use JAX for the `vmap`/linalg examples (13–15). JAX isn't installed
> here, so those use the **torch** backend instead (`einx.torch.adapt_with_vmap`). The
> einx string syntax is identical regardless of backend.

In [2]:
import numpy as np
import torch
import einx

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)
print("einx", einx.__version__)


def check(einx_out, plain_out, tol=1e-6):
    a = np.asarray(einx_out)
    b = np.asarray(plain_out)
    assert a.shape == b.shape, f"shape mismatch: {a.shape} vs {b.shape}"
    assert np.allclose(a, b, atol=tol), "VALUES DIFFER"
    print(f"einx == plain  ✓   shape = {a.shape}")

einx 0.4.2


## 1. Matrix multiplication — `einx.dot`

`"a [b], [b] c -> a c"` — `b` is bracketed in both inputs, so it's the **contraction**
axis; `a` and `c` are kept.

In [3]:
x = np.arange(6.).reshape(2, 3)    # (a, b)
y = np.arange(12.).reshape(3, 4)   # (b, c)

z_einx  = einx.dot("a [b], [b] c -> a c", x, y)
z_plain = x @ y                    # plain: contract shared inner dim

print("einx:\n", z_einx)
print("plain:\n", z_plain)
check(z_einx, z_plain)

einx:
 [[20. 23. 26. 29.]
 [56. 68. 80. 92.]]
plain:
 [[20. 23. 26. 29.]
 [56. 68. 80. 92.]]
einx == plain  ✓   shape = (2, 4)


## 2. Element-wise addition — `einx.add`

`"a b, a b -> a b"` — identical patterns, nothing bracketed: pure element-wise add.

In [4]:
x = np.array([[1., 2., 3.], [4., 5., 6.]])
y = np.array([[10., 20., 30.], [40., 50., 60.]])

z_einx  = einx.add("a b, a b -> a b", x, y)
z_plain = x + y

print(z_einx)
check(z_einx, z_plain)

[[11. 22. 33.]
 [44. 55. 66.]]
einx == plain  ✓   shape = (2, 3)


## 3. Transposed addition — `einx.add`

`"a b, b a -> a b"` — the second operand's axes are named in the *opposite* order, so
einx transposes it before adding. No manual `.T` needed.

In [5]:
x = np.array([[1., 2., 3.], [4., 5., 6.]])         # (a, b) = (2, 3)
y = np.array([[10., 40.], [20., 50.], [30., 60.]]) # (b, a) = (3, 2)

z_einx  = einx.add("a b, b a -> a b", x, y)
z_plain = x + y.T                                  # plain: transpose y to (a, b)

print(z_einx)
check(z_einx, z_plain)

[[11. 22. 33.]
 [44. 55. 66.]]
einx == plain  ✓   shape = (2, 3)


## 4. Outer product — `einx.multiply`

`"a, b -> a b"` — two vectors with *different* axis names produce a 2-D grid: each
becomes a new dimension of the output (broadcasting).

In [6]:
x = np.array([1., 2., 3.])         # (a,)
y = np.array([10., 20., 30., 40.]) # (b,)

z_einx  = einx.multiply("a, b -> a b", x, y)
z_plain = np.outer(x, y)           # == x[:, None] * y[None, :]

print(z_einx)
check(z_einx, z_plain)

[[ 10.  20.  30.  40.]
 [ 20.  40.  60.  80.]
 [ 30.  60.  90. 120.]]
einx == plain  ✓   shape = (3, 4)


## 5. Broadcasting a scalar — `einx.add`

`"a b, -> a b"` — the empty pattern between the comma and `->` is a **scalar** (0-D)
operand, broadcast over the whole matrix.

In [6]:
x = np.ones((2, 3))
y = np.array(5.)                   # 0-D scalar

z_einx  = einx.add("a b, -> a b", x, y)
z_plain = x + y

print(z_einx)
check(z_einx, z_plain)

[[6. 6. 6.]
 [6. 6. 6.]]
einx == plain  ✓   shape = (2, 3)


## 6. Broadcasting a vector along an axis — `einx.add`

`"a b, a -> a b"` — `y` has only the `a` axis, so it's broadcast across `b`. einx
inserts the missing axis for you (the plain form needs an explicit `[:, None]`).

In [7]:
x = np.zeros((2, 3))
y = np.array([10., 20.])           # (a,)

z_einx  = einx.add("a b, a -> a b", x, y)
z_plain = x + y[:, None]           # add new axis so y broadcasts over b

print(z_einx)
check(z_einx, z_plain)

[[10. 10. 10.]
 [20. 20. 20.]]
einx == plain  ✓   shape = (2, 3)


## 7. Sum reduction over one axis — `einx.sum`

`"a b [c] -> a b"` — the bracketed `c` is the axis that gets reduced away.

In [7]:
x = np.arange(24.).reshape(2, 3, 4)   # (a, b, c)

y_einx  = einx.sum("a b [c] -> a b", x)
y_plain = x.sum(axis=-1)              # axis 2 == c

print(x)
print(y_einx)
check(y_einx, y_plain)

[[[ 0.  1.  2.  3.]
  [ 4.  5.  6.  7.]
  [ 8.  9. 10. 11.]]

 [[12. 13. 14. 15.]
  [16. 17. 18. 19.]
  [20. 21. 22. 23.]]]
[[ 6. 22. 38.]
 [54. 70. 86.]]
einx == plain  ✓   shape = (2, 3)


## 8. Reducing multiple axes — bracket placement is positional

You can bracket several axes. **Which axis survives depends on where the *unbracketed*
name sits in the input pattern**, because names map to dimensions positionally:

- `"[a] b [c] -> b"` reduces dims 0 and 2 (a, c), keeps dim 1.
- `"[a c] b -> b"` names dims 0,1 as the reduced ones and dim 2 as `b`, so it keeps dim 2.

So on the *same* `(2,3,4)` tensor these are **different** reductions — a good reminder
that einx axis names are positional labels, not magic.

In [9]:
x = np.arange(24.).reshape(2, 3, 4)

# Pattern A: keep the middle axis (reduce a=dim0, c=dim2)
yA_einx  = einx.sum("[a] b [c] -> b", x)
yA_plain = x.sum(axis=(0, 2))
print("A '[a] b [c] -> b':", yA_einx, " (keeps dim 1)")
check(yA_einx, yA_plain)

# Pattern B: a,c are the first two dims here, b is the last -> keep dim 2
yB_einx  = einx.sum("[a c] b -> b", x)
yB_plain = x.sum(axis=(0, 1))
print("B '[a c] b -> b':", yB_einx, " (keeps dim 2)")
check(yB_einx, yB_plain)

A '[a] b [c] -> b': [ 60.  92. 124.]  (keeps dim 1)
einx == plain  ✓   shape = (3,)
B '[a c] b -> b': [60. 66. 72. 78.]  (keeps dim 2)
einx == plain  ✓   shape = (4,)


## 9. Transpose / identity — `einx.id`

`einx.id` applies the identity op while rearranging axes — i.e. a transpose.
(`einx.rearrange` would do the same.)

In [10]:
x = np.arange(6.).reshape(2, 3)

y_einx  = einx.id("a b -> b a", x)
y_plain = x.T                       # np.transpose(x, (1, 0))

print(y_einx)
check(y_einx, y_plain)

[[0. 3.]
 [1. 4.]
 [2. 5.]]
einx == plain  ✓   shape = (3, 2)


## 10. Softmax (shape-preserving) — `einx.softmax`

`"a b [c] -> a b [c]"` — `c` is bracketed on both sides: softmax is computed *over* `c`
but the shape is preserved.

In [11]:
x = rng.standard_normal((2, 3, 4))

y_einx = einx.softmax("a b [c] -> a b [c]", x)

# plain: numerically-stable softmax over the last axis
e = np.exp(x - x.max(axis=-1, keepdims=True))
y_plain = e / e.sum(axis=-1, keepdims=True)

print("each row sums to 1:\n", y_einx.sum(-1))
check(y_einx, y_plain)

each row sums to 1:
 [[1. 1. 1.]
 [1. 1. 1.]]
einx == plain  ✓   shape = (2, 3, 4)


## 11. Conditional select — `einx.where`

`"a b, a, b -> a b"` — the condition is full `(a, b)`, while the two value operands are
broadcast from `(a,)` and `(b,)` respectively.

In [12]:
cond = np.array([[True, False, True], [False, True, False]])  # (a, b)
x = np.array([10., 20.])        # (a,) -> used where cond is True
y = np.array([1., 2., 3.])      # (b,) -> used where cond is False

z_einx  = einx.where("a b, a, b -> a b", cond, x, y)
z_plain = np.where(cond, x[:, None], y[None, :])

print(z_einx)
check(z_einx, z_plain)

[[10.  2. 10.]
 [ 1. 20.  3.]]
einx == plain  ✓   shape = (2, 3)


## 12. Gather at coordinates — `einx.get_at`

`"b [h w] c, b p [2] -> b p c"` — for each batch `b` and point `p`, read the pixel at
the `(h, w)` coordinate held in the last (`[2]`) axis of `indices`. The bracketed
`[h w]` are the axes being indexed into.

In [13]:
b, h, w, c, p = 2, 4, 4, 3, 5
image   = np.arange(b * h * w * c).reshape(b, h, w, c).astype(float)
indices = rng.integers(0, 4, size=(b, p, 2))     # (b, p, 2): each row is (h, w)

y_einx = einx.get_at("b [h w] c, b p [2] -> b p c", image, indices)

# plain: advanced-index each batch then stack
y_plain = np.stack([
    image[i, indices[i, :, 0], indices[i, :, 1], :]  # -> (p, c)
    for i in range(b)
])

print("output shape:", y_einx.shape)
check(y_einx, y_plain)

output shape: (2, 5, 3)
einx == plain  ✓   shape = (2, 5, 3)


## 13. Adapt a custom numpy-style reduction — `einx.numpy.adapt_numpylike_reduce`

Wrap any `f(x, axis=...)` reduction so it speaks einx notation.
`"a [b] c -> c a"` reduces `b`, then rearranges the kept axes to `c a`.

In [14]:
x = rng.standard_normal((2, 3, 4))   # (a, b, c)

def myfunc(arr, axis):
    return 0.5 * np.sum(arr ** 2, axis=axis)

einmyfunc = einx.numpy.adapt_numpylike_reduce(myfunc)
y_einx = einmyfunc("a [b] c -> c a", x)

# plain: reduce axis=1 (b) -> (a, c), then transpose to (c, a)
y_plain = (0.5 * np.sum(x ** 2, axis=1)).T

print("shape:", y_einx.shape)
check(y_einx, y_plain)

shape: (4, 2)
einx == plain  ✓   shape = (4, 2)


## 14. Adapt an arbitrary op via vmap — `einx.torch.adapt_with_vmap`

Instead of a reduction adapter, vectorize a plain elementary function with `vmap`. The
function receives only the **bracketed** axes; einx vmaps over the rest. Same result as
#13, computed by vmapping over the `(a, c)` axes.

In [15]:
x = torch.from_numpy(rng.standard_normal((2, 3, 4)))   # (a, b, c)

def half_sq_sum(v):       # v has shape (b,) -> returns a scalar
    return 0.5 * torch.sum(v ** 2)

einmyfunc = einx.torch.adapt_with_vmap(half_sq_sum)
y_einx = einmyfunc("a [b] c -> c a", x)

# plain: 0.5 * sum over b -> (a, c), transpose -> (c, a)
y_plain = (0.5 * (x ** 2).sum(dim=1)).T

print("shape:", tuple(y_einx.shape))
check(y_einx, y_plain)

shape: (4, 2)
einx == plain  ✓   shape = (4, 2)


## 15. Batched linear algebra via vmap — `einx.torch.adapt_with_vmap`

vmap turns single-matrix routines into batched ones. The two matrix dims must get
**distinct** names (`n n2`) even though they're equal length. Builtins are wrapped in a
`def` so einx can introspect their signature.

In [16]:
# Build a batch of SPD (hence invertible) matrices.
M = rng.standard_normal((4, 3, 3))
A = torch.from_numpy(M @ np.transpose(M, (0, 2, 1)) + 3 * np.eye(3))   # (a, n, n)
bvec = torch.from_numpy(rng.standard_normal((4, 3)))                   # (a, n)

# --- batched solve ---
def solve(Mat, vec):
    return torch.linalg.solve(Mat, vec)

einsolve = einx.torch.adapt_with_vmap(solve)
x_einx  = einsolve("a [n n2], a [n] -> a [n]", A, bvec)
x_plain = torch.linalg.solve(A, bvec)     # torch already batches natively
print("solve shape:", tuple(x_einx.shape))
check(x_einx, x_plain, tol=1e-4)

# --- batched determinant ---
def det(Mat):
    return torch.linalg.det(Mat)

eindet = einx.torch.adapt_with_vmap(det)
d_einx  = eindet("a [n n2] -> a", A)
d_plain = torch.linalg.det(A)
print("det:", d_einx)
check(d_einx, d_plain, tol=1e-4)

solve shape: (4, 3)
einx == plain  ✓   shape = (4, 3)


det: tensor([161.4973, 162.2487, 173.0374,  69.3447], dtype=torch.float64)
einx == plain  ✓   shape = (4,)


## Takeaways

- The string is the whole API: name axes, bracket the ones the op acts on, list outputs.
- Names are **positional** per tensor; matching names across tensors = same axis.
- Unbracketed axes are auto-vectorized (broadcast or vmapped) — that's what removes the
  manual `[:, None]`, `.T`, `axis=...`, and per-batch loops in the plain versions.